# 14 · Results: performance, per-class skill, reliability and model selection (5-class)

Everything here is computed from the outer-fold predictions of notebook 13: Downtown development boreholes, each predicted by a model that never saw it, its fold or its labels during tuning.

**How to read the numbers.** Five classes are a harder task than Clay/Sand, so scores are lower than binary scores and cannot be compared with them directly. Judge each model against:

* **chance**, shown below for the observed class shares;
* **the depth-only reference** (`DEPTH_PRIOR`: class shares of the depth slice), which shows what location adds;
* **Cohen's kappa**, which is agreement beyond chance (0 = chance, 1 = perfect);
* **the merged Clay/Sand check** (`bin_*`): the same 5-class predictions collapsed to Clay/Sand.

In [ ]:
import os, sys, json
from pathlib import Path

# 5-class soil model: this line selects config/project_5class.toml (keep it).
os.environ["SIEMREAP_CONFIG"] = "project_5class.toml"
# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner(), "|", cfg["_config_name"])

from siemreap import metrics as M, folds as F
from siemreap import data as D, plots as PL
CLS = D.class_info(cfg)["classes"]
NAMES = [c["name"] for c in CLS]
COLOURS = PL.class_colours(CLS)
K = len(NAMES)
AGG = config.stage_dir(cfg, "04_results", "aggregate")
RES = config.stage_dir(cfg, "04_results", "restricted")
preds = pd.read_csv(config.stage_dir(cfg, "03_nested_cv", "restricted") / "outer_predictions.csv")
primary = F.primary_block(cfg) if MODE == "full" else int(preds.block_m.iloc[0])
main = preds[(preds.block_m == primary) & (preds.purge_m == 0)]
print(f"{len(preds):,} outer-fold predictions; {preds.groupby(M.RUN_KEYS).ngroups} runs; primary block {primary} m")
pd.set_option("display.max_columns", 40)

## 1. Chance levels for these class shares

In [ ]:
one = main[main.model == main.model.iloc[0]]
pi = one.y.value_counts(normalize=True).reindex(range(K), fill_value=0).values
chance = {"always the largest class: accuracy": pi.max(),
          "random by class shares: accuracy": float((pi ** 2).sum()),
          "random by class shares: Macro-F1": float(pi.mean()),
          "uniform random: Macro-F1": float(np.mean(2 * pi / K / (pi + 1 / K))),
          "any chance prediction: kappa": 0.0}
display(pd.Series(chance).round(3).to_frame("value"))
display(pd.DataFrame({"class": NAMES, "share of test samples": pi.round(3)}))

## 2. Outer-fold metrics (mean ± SD over the 5 folds) and pooled out-of-fold metrics

In [ ]:
folds_t = M.fold_table(preds)
summary = M.fold_summary(folds_t)
pooled = M.pooled_table(preds)
folds_t.to_csv(AGG / "metrics_by_outer_fold.csv", index=False)
summary.to_csv(AGG / "metrics_fold_mean_sd.csv", index=False)
pooled.to_csv(AGG / "metrics_pooled_oof.csv", index=False)
show = ["block_m", "model", "accuracy_mean", "macro_f1_mean", "macro_f1_sd", "macro_f1_min", "kappa_mean",
        "balanced_accuracy_mean", "roc_auc_ovr_mean", "log_loss_mean", "brier_mean", "bin_accuracy_mean", "bin_macro_f1_mean"]
display(summary[show].round(3).sort_values(["block_m", "macro_f1_mean"], ascending=[True, False]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=150)
from siemreap import plots
plots.fold_dotplot(folds_t[folds_t.block_m == primary], "macro_f1", f"Macro-F1 by outer fold, {primary} m blocks", ax=axes[0])
plots.fold_dotplot(folds_t[folds_t.block_m == primary], "kappa", f"Cohen's kappa by outer fold, {primary} m blocks", ax=axes[1])
fig.tight_layout(); fig.savefig(AGG / "fig_outer_fold_metrics.png", bbox_inches="tight")

### Difference from the depth-only reference, fold by fold (same test samples)

In [ ]:
f = folds_t[folds_t.purge_m == 0]
ref = f[f.model == "DEPTH_PRIOR"].set_index(["block_m", "outer_fold"])
rows = []
for (b, m), g in f[f.model != "DEPTH_PRIOR"].groupby(["block_m", "model"]):
    g = g.set_index(["block_m", "outer_fold"])
    for met in ("macro_f1", "kappa", "accuracy", "log_loss"):
        d = g[met] - ref.loc[g.index, met]
        rows.append({"block_m": b, "model": m, "metric": met, "mean_diff": d.mean(), "sd": d.std(ddof=1),
                     "folds_better": int((d < 0).sum() if met == "log_loss" else (d > 0).sum()), "folds": len(d)})
vs_ref = pd.DataFrame(rows).round(3)
display(vs_ref.pivot_table(index=["block_m", "model"], columns="metric", values=["mean_diff", "folds_better"]))
vs_ref.to_csv(AGG / "difference_from_depth_reference.csv", index=False)

## 3. Per-class skill (as Khamjan et al. 2026, Table 4 and Fig. 4)

In [ ]:
pc = summary[summary.block_m == primary].set_index("model")
f1_tab = pc[[f"f1_c{c}_mean" for c in range(K)]].set_axis(NAMES, axis=1).round(3)
rec_tab = pc[[f"recall_c{c}_mean" for c in range(K)]].set_axis(NAMES, axis=1).round(3)
print("F1 by class (mean over outer folds)"); display(f1_tab)
print("Recall by class (mean over outer folds)"); display(rec_tab)
f1_tab.to_csv(AGG / "f1_by_class.csv"); rec_tab.to_csv(AGG / "recall_by_class.csv")

In [ ]:
models = [m for m in plots.MODEL_ORDER if m in set(main.model)]
fig, axes = plt.subplots(2, len(models), figsize=(4.6 * len(models), 8.6), dpi=130, squeeze=False)
pr_rows = []
for j, m in enumerate(models):
    g = main[main.model == m]
    plots.confusion_plot(M.confusion_table(g, K), NAMES, title=f"{m}: confusion (% recall)", ax=axes[0, j])
    plots.pr_curves(g, NAMES, COLOURS, title=f"{m}: precision-recall", ax=axes[1, j])
    pr_rows.append(M.pr_auc_by_class(g).assign(model=m))
fig.tight_layout(); fig.savefig(AGG / "fig_confusion_and_pr_by_model.png", bbox_inches="tight")
pr = pd.concat(pr_rows); pr["class"] = pr["class"].map(dict(enumerate(NAMES)))
display(pr.pivot_table(index="model", columns="class", values="pr_auc", sort=False).round(3))
display(pr.drop_duplicates("class")[["class", "baseline"]].round(3))
pr.to_csv(AGG / "pr_auc_by_class.csv", index=False)

## 4. Merged back to Clay/Sand

In [ ]:
display(summary[["block_m", "model", "bin_accuracy_mean", "bin_macro_f1_mean", "bin_recall_clay_mean", "bin_recall_sand_mean"]]
        .round(3).sort_values(["block_m", "bin_macro_f1_mean"], ascending=[True, False]))

## 5. Depth bands and distance to the nearest training borehole

In [ ]:
depth_t = M.depth_band_table(preds, cfg["depth"]["bands_m"]); depth_t.to_csv(AGG / "metrics_by_depth_band.csv", index=False)
dist_t = M.distance_table(preds); dist_t.to_csv(AGG / "metrics_by_distance_to_training.csv", index=False)
sel_runs = lambda t: t[(t.block_m == primary) & (t.purge_m == 0)]
for tab, key in ((depth_t, "depth_band"), (dist_t, "distance_bin")):
    display(sel_runs(tab).pivot_table(index="model", columns=key, values="macro_f1", observed=True).round(3))
    display(sel_runs(tab).drop_duplicates(key)[[key, "n"]].set_index(key).T)
axes = plots.band_plot(sel_runs(depth_t), "depth_band", "macro_f1", "Macro-F1 by depth band")
axes[0].figure.savefig(AGG / "fig_macro_f1_by_depth_band.png", bbox_inches="tight")
axes = plots.band_plot(sel_runs(dist_t), "distance_bin", "macro_f1", "Macro-F1 by distance to the nearest training borehole")
axes[0].figure.savefig(AGG / "fig_macro_f1_by_distance.png", bbox_inches="tight")

## 6. Calibration and borehole-level results

In [ ]:
rel = M.reliability_table(main); rel.to_csv(AGG / "reliability_top_label.csv", index=False)
fig, ax = plt.subplots(figsize=(5, 4.6), dpi=150)
plots.reliability_plot(rel, title="Top-label reliability (confidence vs accuracy)", ax=ax)
ax.set_xlabel("confidence of the predicted class"); ax.set_ylabel("share correct")
fig.savefig(AGG / "fig_reliability.png", bbox_inches="tight")
bh_t = M.borehole_table(preds); bh_t.to_csv(RES / "metrics_by_borehole.csv", index=False)
agg_bh = (bh_t.groupby(M.RUN_KEYS).agg(boreholes=("BH_Code", "size"), accuracy_median=("accuracy", "median"),
          accuracy_p10=("accuracy", lambda s: s.quantile(0.1)), bin_accuracy_median=("bin_accuracy", "median"))
          .round(3).reset_index())
agg_bh.to_csv(AGG / "borehole_level_summary.csv", index=False); display(agg_bh)

In [ ]:
best = summary[(summary.block_m == primary) & (summary.model != "DEPTH_PRIOR")].sort_values("macro_f1_mean").iloc[-1].model
pr_ = main[main.model == best]
codes = pr_.drop_duplicates("BH_Code").sort_values(["outer_fold", "dist_nearest_train_m"]).BH_Code.head(24)
fig = plots.profiles_mc(pr_, codes, COLOURS, bottom_m=cfg["depth"]["bottom_m"], title=f"Observed vs predicted class, {best}, {primary} m blocks")
fig.savefig(RES / "fig_profiles_example.png", bbox_inches="tight")

## 7. Pre-declared model selection

Rule (design report section 11), applied to the primary block size:

1. **Eligible:** the mean recall of *every* class is at least 1/K, so no class is ignored.
2. **Rank** by mean outer-fold Macro-F1.
3. **Choose** among eligible models within one standard error of the best: the lowest fold SD, then the lowest log loss.

In [ ]:
sel = M.select_model_mc(summary, cfg, primary)
display(sel["table"][["model", "eligible", "macro_f1_mean", "macro_f1_sd", "kappa_mean", "min_class_recall_mean", "log_loss_mean"]]
        .round(3).sort_values("macro_f1_mean", ascending=False))
record = {k: v for k, v in sel.items() if k != "table"}
record["mode"] = MODE
(AGG / "selected_model.json").write_text(json.dumps(record, indent=1, default=str))
print(json.dumps(record, indent=1, default=str))